In [ ]:
%pip install openai

# Workload Profiler — Pay-Per-Token

Run sample queries against a PPT endpoint and measure the four metrics needed to size a PT endpoint:

| Metric | Description |
|---|---|
| **Input tokens** | Prompt tokens consumed per request |
| **Output tokens** | Completion tokens generated per request |
| **TTFT** | Time to First Token — latency from request start to first streamed token |
| **TPOT** | Time Per Output Token — avg time between tokens after the first |

TTFT and TPOT require streaming — the client measures wall-clock time between chunk arrivals.

Use the output to populate the [Databricks GenAI Pricing Calculator](https://www.databricks.com/product/pricing/genai-pricing-calculator) with your real workload shape.

In [ ]:
import os

os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Change to the pay-per-token model you want to profile
PPT_MODEL = "databricks-gpt-oss-20b"

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints",
    api_key=os.environ["DATABRICKS_TOKEN"],
)

In [ ]:
import time


def profile_request(messages: list[dict]) -> dict:
    """
    Two calls per query:
    1. Non-streaming  → input/output token counts (usage is in the response object)
    2. Streaming      → TTFT and TPOT (Databricks API does not support stream_options)

    TTFT = time from request start → first token received
    TPOT = (total_time - TTFT) / (output_tokens - 1)
    """
    # --- token counts ---
    response = client.chat.completions.create(
        model=PPT_MODEL,
        messages=messages,
    )
    input_tokens  = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens

    # --- latency metrics ---
    ttft_ms = None
    t_start = time.perf_counter()

    stream = client.chat.completions.create(
        model=PPT_MODEL,
        messages=messages,
        stream=True,
    )

    for chunk in stream:
        if ttft_ms is None and chunk.choices and chunk.choices[0].delta.content:
            ttft_ms = (time.perf_counter() - t_start) * 1000

    total_ms = (time.perf_counter() - t_start) * 1000
    tpot_ms  = (total_ms - ttft_ms) / (output_tokens - 1) if output_tokens > 1 else 0.0

    return {
        "input_tokens":  input_tokens,
        "output_tokens": output_tokens,
        "ttft_ms":       round(ttft_ms, 2),
        "tpot_ms":       round(tpot_ms, 2),
        "total_ms":      round(total_ms, 2),
    }

## Sample queries

Replace or extend `SAMPLE_QUERIES` with prompts representative of your actual workload.
The goal is to capture a realistic distribution of input and output lengths.

In [ ]:
SAMPLE_QUERIES = [
    "What is machine learning? Explain in 3 paragraphs.",
    "Summarize the key differences between supervised and unsupervised learning.",
    "Write a Python function that reads a CSV file and returns the top 5 rows.",
    "What is the capital of France?",
    "Explain transformer architecture in detail, covering attention mechanisms, positional encoding, and the encoder-decoder structure.",
]

results = []

for i, query in enumerate(SAMPLE_QUERIES):
    print(f"[{i+1}/{len(SAMPLE_QUERIES)}] {query[:60]}...")
    result = profile_request([{"role": "user", "content": query}])
    results.append({"query": query, **result})
    print(f"  input={result['input_tokens']} output={result['output_tokens']} "
          f"ttft={result['ttft_ms']}ms tpot={result['tpot_ms']}ms\n")

In [ ]:
import statistics

def percentile(data: list[float], p: int) -> float:
    sorted_data = sorted(data)
    idx = int(len(sorted_data) * p / 100)
    return round(sorted_data[min(idx, len(sorted_data) - 1)], 2)

def summarize(key: str) -> dict:
    vals = [r[key] for r in results]
    return {
        "avg":  round(statistics.mean(vals), 2),
        "p50":  percentile(vals, 50),
        "p75":  percentile(vals, 75),
        "p95":  percentile(vals, 95),
        "max":  round(max(vals), 2),
    }

metrics = ["input_tokens", "output_tokens", "ttft_ms", "tpot_ms"]
labels  = ["Input tokens", "Output tokens", "TTFT (ms)", "TPOT (ms)"]

print(f"{'Metric':<18} {'avg':>8} {'p50':>8} {'p75':>8} {'p95':>8} {'max':>8}")
print("-" * 58)
for metric, label in zip(metrics, labels):
    s = summarize(metric)
    print(f"{label:<18} {s['avg']:>8} {s['p50']:>8} {s['p75']:>8} {s['p95']:>8} {s['max']:>8}")

print(f"\nModel: {PPT_MODEL}  |  n={len(results)} requests")
print("\nUse avg input/output tokens and QPM from your production traffic")
print("in the Databricks GenAI Pricing Calculator to size your PT endpoint.")